# Kaggle Submission — March Machine Learning Mania 2026

In [1]:
import warnings
import joblib
import numpy as np
import pandas as pd

warnings.simplefilter(action='ignore', category=UserWarning)

In [2]:
# Load trained model
model = joblib.load('model/march_madness_model_W.joblib')

# Load season stats and reconstruct feature_cols the same way 2_train_2026 does
final = (
    pd.read_csv('data_2026/final_features_W.csv')
    .rename(columns={
        'WTEAMID': 'W_TEAMID',
        'LTEAMID': 'L_TEAMID',
        'WSCORE':  'W_SCORE',
        'LSCORE':  'L_SCORE',
    })
)
final['WIN_INDICATOR'] = 1
drop_cols = ['W_CTWINS', 'W_AVERAGECTSCORE', 'L_CTWINS', 'L_AVERAGECTSCORE',
             'W_WLOCN', 'W_WLOCH', 'W_WLOCA', 'L_WLOCN', 'L_WLOCH', 'L_WLOCA']
final = final.drop(columns=[c for c in drop_cols if c in final.columns])

NON_FEATURE_COLS = {'SEASON', 'WIN_INDICATOR', 'L_TEAMID', 'W_TEAMID',
                    'W_SCORE', 'L_SCORE', 'ROUND', 'L_REGION', 'W_REGION'}
feature_cols = [c for c in final.columns if c not in NON_FEATURE_COLS]
print(f'Feature columns: {len(feature_cols)}')

Feature columns: 157


/var/folders/fn/v37r1g5j74928hgvsvhgtlhh0000gn/T/ipykernel_54446/742542480.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final['WIN_INDICATOR'] = 1


In [3]:
# Load 2026 season stats
BRACKET_SEASON = 2026
season_stats_all = pd.read_csv('data_2026/final_season_stats_W.csv')
season_stats = season_stats_all[season_stats_all.SEASON == BRACKET_SEASON].drop(columns=['SEASON'])
season_stats = season_stats.drop(columns=[c for c in ['REGION'] if c in season_stats.columns])
print(f'Teams with 2026 season stats: {len(season_stats)}')

Teams with 2026 season stats: 359


In [4]:
# Read sample submission — defines all matchup IDs we need to predict
sample_sub = pd.read_csv('data_2026/SampleSubmissionStage2.csv')
print(f'Sample submission rows: {len(sample_sub)}')
print(sample_sub.head())

Sample submission rows: 132133
               ID  Pred
0  2026_1101_1102   0.5
1  2026_1101_1103   0.5
2  2026_1101_1104   0.5
3  2026_1101_1105   0.5
4  2026_1101_1106   0.5


In [5]:
# Parse IDs into season, team1, team2 — predict all women's rows
# Women's team IDs are in the 3000s range
id_parts = sample_sub['ID'].str.split('_', expand=True)
sample_sub['TEAM1'] = id_parts[1].astype(int)
sample_sub['TEAM2'] = id_parts[2].astype(int)

# Filter to women's matchups only (team IDs >= 3000)
matchups = sample_sub[
    (sample_sub['TEAM1'] >= 3000) & (sample_sub['TEAM2'] >= 3000)
].copy()
print(f"Women's matchups to predict: {len(matchups)}")

Women's matchups to predict: 65703


In [6]:
# Join season stats for team1 (as W_) and team2 (as L_)
# Teams without 2026 stats get their feature values filled with the column median
stats_w = season_stats.copy()
stats_w.columns = ['W_' + c for c in stats_w.columns]

stats_l = season_stats.copy()
stats_l.columns = ['L_' + c for c in stats_l.columns]

matchups = matchups.merge(stats_w, left_on='TEAM1', right_on='W_TEAMID', how='left')
matchups = matchups.merge(stats_l, left_on='TEAM2', right_on='L_TEAMID', how='left')

# Fill missing feature values with column median (teams not in 2026 season stats)
for col in feature_cols:
    if col not in matchups.columns:
        matchups[col] = 0
    elif matchups[col].isna().any():
        matchups[col] = matchups[col].fillna(matchups[col].median())

missing = matchups[feature_cols].isna().sum().sum()
teams_missing_stats = matchups['W_TEAMID'].isna().sum()
print(f'Rows with missing team stats (filled with median): {teams_missing_stats}')
print(f'Remaining missing feature values: {missing}')

Rows with missing team stats (filled with median): 523
Remaining missing feature values: 0


In [7]:
# Predict probability that TEAM1 beats TEAM2
# predict_proba returns [P(loss), P(win)] — take column index 1
proba = model.predict_proba(matchups[feature_cols])
matchups['Pred'] = proba[:, 1]

print(f'Prediction range: {matchups.Pred.min():.4f} – {matchups.Pred.max():.4f}')
print(matchups[['ID', 'Pred']].head(10))

Prediction range: 0.0025 – 0.9963
               ID      Pred
0  2026_3101_3102  0.523085
1  2026_3101_3103  0.456258
2  2026_3101_3104  0.114652
3  2026_3101_3105  0.519510
4  2026_3101_3106  0.554504
5  2026_3101_3107  0.339372
6  2026_3101_3108  0.693135
7  2026_3101_3110  0.438557
8  2026_3101_3111  0.171428
9  2026_3101_3112  0.502414


In [8]:
# Output submission — every row has a real prediction
submission = matchups[['ID', 'Pred']].copy()
submission.to_csv('submission_2026_W.csv', index=False)
print(f'Saved submission_2026.csv ({len(submission)} rows)')
print(f'Pred range: {submission.Pred.min():.4f} – {submission.Pred.max():.4f}')
print(submission.head(10))

Saved submission_2026.csv (65703 rows)
Pred range: 0.0025 – 0.9963
               ID      Pred
0  2026_3101_3102  0.523085
1  2026_3101_3103  0.456258
2  2026_3101_3104  0.114652
3  2026_3101_3105  0.519510
4  2026_3101_3106  0.554504
5  2026_3101_3107  0.339372
6  2026_3101_3108  0.693135
7  2026_3101_3110  0.438557
8  2026_3101_3111  0.171428
9  2026_3101_3112  0.502414
